# 11 -- Deployment Results

Parses the A100 torchao int8-vs-fp32 throughput diagnosis log (a plain-text benchmark
report, not a CSV -- no structured deployment-throughput CSV exists in the results tree;
see report Sec. 5.10 for why) and extracts the per-combination mean speedup for the two
torchao INT8 configurations. Reproduces Table 10 of the report.

Source (log, not CSV): `results/cluster/20260807_100652_27816/logs/int8_perf_diagnosis.txt`


In [1]:
# Requirements: pandas==3.0.5, numpy==2.5.1, matplotlib==3.11.1, seaborn==0.13.2, scipy==1.18.0
# All notebooks in this report use the same environment; paths below are relative to
# report/notebooks/, so the notebook must be run with its own directory as the working
# directory (the default for `jupyter nbconvert --execute` and for Jupyter's own kernel).
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

REPO = "../.."  # report/notebooks -> report -> repo root
FIG_DIR = "../figures"
import os
os.makedirs(FIG_DIR, exist_ok=True)


In [2]:
import re

LOG_PATH = f"{REPO}/results/cluster/20260807_100652_27816/logs/int8_perf_diagnosis.txt"
with open(LOG_PATH) as f:
    log_text = f.read()
print(log_text[:600])


torchao int8 vs fp32 performance diagnosis
GPU: NVIDIA A100-SXM4-80GB
Compute capability: 8.0 (sm80)

torchao 0.17 int8 Conv2d support: Int8DynamicActivationInt8WeightConfig's default
filter_fn only matches nn.Linear (aten.linear/aten.mm dispatch); it does not
implement aten.conv2d at all, so Conv2d layers are left untouched fp32 under
quantize_(model, Int8DynamicActivationInt8WeightConfig()) with no filter_fn
override. The only config that quantizes Conv2d weights, IntxWeightOnlyConfig, is
weight-only: it dequantiz


The log is organized into `MODEL / DATASET / STAGE` section headers followed eventually
by a `Mean speedup_x across batch sizes: int8_weight_only=X, int8_dynamic_act=Y` line.
Parse both out with a regex per section rather than hand-copying the numbers, so this
table stays reproducible if the log is regenerated.


In [3]:
section_re = re.compile(
    r"^(\S+) / (\S+) / (\S+)\s*\n={5,}\n(.*?)(?=\n={5,}\n\S+ / \S+ / \S+|\Z)",
    re.MULTILINE | re.DOTALL,
)
speedup_re = re.compile(
    r"Mean speedup_x across batch sizes: int8_weight_only=([\d.]+), int8_dynamic_act=([\d.]+)"
)

rows = []
for model, dataset, stage, body in section_re.findall(log_text):
    m = speedup_re.search(body)
    if not m:
        continue
    rows.append({
        "model": model, "dataset": dataset, "stage": stage,
        "int8_weight_only": float(m.group(1)), "int8_dynamic_act": float(m.group(2)),
    })

deploy = pd.DataFrame(rows)
deploy


,model,dataset,stage,int8_weight_only,int8_dynamic_act
0,cnn,CIFAR10,PTQ,1.042,0.209
1,cnn,CIFAR10,QAT,1.053,0.210
2,resnet18_no_weights,CIFAR10,PTQ,1.190,0.558
3,resnet18_no_weights,CIFAR10,QAT,1.195,0.559
4,resnet50_no_weights,CIFAR10,PTQ,1.116,0.645
5,resnet50_no_weights,CIFAR10,QAT,1.116,0.647
6,cnn,IMAGENET100,PTQ,1.155,0.574
7,cnn,IMAGENET100,QAT,1.155,0.575
8,resnet18_no_weights,IMAGENET100,PTQ,1.121,0.611
9,resnet18_no_weights,IMAGENET100,QAT,1.121,0.612


Average PTQ and QAT mean-speedups per (model, dataset) -- the report table reports one row per combination, not split by stage, since the two stages' speedups are near-identical (both start from the same architecture; PoT-vs-FP32 arithmetic cost does not depend on how the checkpoint was produced).


In [4]:
MODEL_LABEL = {"cnn": "CNN", "resnet18_no_weights": "ResNet-18", "resnet50_no_weights": "ResNet-50"}
DATASET_LABEL = {"IMAGENET100": "ImageNet100", "CIFAR10": "CIFAR10"}

table10 = deploy.groupby(["model", "dataset"])[["int8_weight_only", "int8_dynamic_act"]].mean().reset_index()
table10["Modell"] = table10["model"].map(MODEL_LABEL)
table10["Datensatz"] = table10["dataset"].map(DATASET_LABEL)
table10 = table10[["Datensatz", "Modell", "int8_weight_only", "int8_dynamic_act"]]
table10.columns = ["Datensatz", "Modell", "wt.-only", "dyn.-akt."]
table10[["wt.-only", "dyn.-akt."]] = table10[["wt.-only", "dyn.-akt."]].round(2)
table10 = table10.sort_values(["Datensatz", "Modell"], ascending=[False, True]).reset_index(drop=True)
table10.to_csv(f"{FIG_DIR}/tab_10_deployment_results.csv", index=False)
table10


,Datensatz,Modell,wt.-only,dyn.-akt.
0,ImageNet100,CNN,1.16,0.57
1,ImageNet100,ResNet-18,1.12,0.61
2,ImageNet100,ResNet-50,1.07,0.67
3,CIFAR10,CNN,1.05,0.21
4,CIFAR10,ResNet-18,1.19,0.56
5,CIFAR10,ResNet-50,1.12,0.65


## Output

- `figures/tab_10_deployment_results.csv` -- report Table 10

Note: the CPU/fbgemm path (Sec. 5.10) is *not* included here -- no completed throughput
CSV or log summary for it exists in the results tree (only calibration/layer-conversion
confirmation messages), so no number is reported for it, per the report's data-provenance
constraint (every number must trace to a concrete source).
